dataset yang di gunakan : https://github.com/indolem/indolem/tree/main/ner

# import file

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datasets import load_dataset
import ast
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [ ]:
import os
import pandas as pd

def read_ner_tsv(file_path):
    sentences = []
    labels = []

    tokens = []
    ner_tags = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if line == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(ner_tags)

                    tokens = []
                    ner_tags = []
            else:
                split = line.split("\t")
                tokens.append(split[0])
                ner_tags.append(split[1])

    return sentences, labels

def combine_datasets(folder_path, prefix):
    all_sentences = []
    all_labels = []

    for i in range(1, 6):
        file_name = f"{prefix}.0{i}.tsv"
        file_path = os.path.join(folder_path, file_name)

        sentences, labels = read_ner_tsv(file_path)

        all_sentences.extend(sentences)
        all_labels.extend(labels)

    return all_sentences, all_labels

folder = "data/indolem/ner/data/nerugm"

train_sentences, train_labels = combine_datasets(folder, "train")
test_sentences, test_labels = combine_datasets(folder, "test")
dev_sentences, dev_labels = combine_datasets(folder, "dev")

train_df = pd.DataFrame({
    "tokens": train_sentences,
    "ner_tags": train_labels
})

test_df = pd.DataFrame({
    "tokens": test_sentences,
    "ner_tags": test_labels
})

dev_df = pd.DataFrame({
    "tokens": dev_sentences,
    "ner_tags": dev_labels
})

train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)
dev_df.to_csv("dev_data.csv", index=False)

In [ ]:
train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')
dev_data = pd.read_csv('dev_data.csv')

In [ ]:
print(f'total sentence train: {len(train_data)}')
print(f'total sentence test: {len(test_data)}')
print(f'total sentence dev: {len(dev_data)}')

total sentence train: 8437
total sentence test: 2343
total sentence dev: 935


# mapping

In [ ]:
LABELS = [
    "O",
    "B-PERSON",    "I-PERSON",
    "B-ORGANIZATION", "I-ORGANIZATION",
    "B-LOCATION",  "I-LOCATION",
    "B-TIME",      "I-TIME",
    "B-QUANTITY",  "I-QUANTITY",
]

In [ ]:
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for idx, label in enumerate(LABELS)}

num_labels = len(LABELS)

print("mapping")
for label, idx in label2id.items():
    print(f'{idx:>2} {label}')
print(f'\ntotal labels: {num_labels}')

device = torch.device("cuda" if torch.cuda.is_available() else "tidak ada cuda")
print(f"device yang digunakan: {device}")

mapping
 0 O
 1 B-PERSON
 2 I-PERSON
 3 B-ORGANIZATION
 4 I-ORGANIZATION
 5 B-LOCATION
 6 I-LOCATION
 7 B-TIME
 8 I-TIME
 9 B-QUANTITY
10 I-QUANTITY

total labels: 11
device yang digunakan: cuda


# tokenisasi

In [ ]:
model = 'bert-base-multilingual-cased'
max_len = 128
batch_size = 16


tokenisasi = BertTokenizerFast.from_pretrained(model)

def parse_df(data):
    sentence = [ast.literal_eval(i) for i in data['tokens']]
    tags = [ast.literal_eval(i) for i in data['ner_tags']]
    return sentence, tags

train_sentences, train_labels = parse_df(train_data)
test_sentences, test_labels = parse_df(test_data)
dev_sentences, dev_labels = parse_df(dev_data)

print(f'train : {len(train_sentences)} sentences')
print(f'test : {len(test_sentences)} sentences')
print(f'dev : {len(dev_sentences)} sentences')

train : 8437 sentences
test : 2343 sentences
dev : 935 sentences


In [ ]:
print('\n contoh kalima dan labelnya')
for token, tag in zip(train_sentences[0], train_labels[0]):
    print(f' {token} {tag}')


 contoh kalima dan labelnya
 Musababnya O
 , O
 dua O
 tafsir O
 mengenai O
 mekanisme O
 pilkada O
 , O
 baik O
 langsung O
 atau O
 pun O
 melalui O
 DPRD O
 sama O
 - O
 sama O
 tidak O
 melanggar O
 konstitusi O
 . O


# classes mengubah kata menjadi tensor



In [ ]:
class NERdataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len, label2id):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            sentence,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        word_ids = encoding.word_ids()
        label_ids = []

        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(self.label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word_id = word_id

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'token_type_ids': encoding['token_type_ids'].squeeze(),
            'labels': torch.tensor(label_ids, dtype=torch.long)
        }

train_dataset = NERdataset(train_sentences, train_labels, tokenisasi, max_len, label2id)
test_dataset = NERdataset(test_sentences, test_labels, tokenisasi, max_len, label2id)
dev_dataset = NERdataset(dev_sentences, dev_labels, tokenisasi, max_len, label2id)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)

print(f'train batch : {len(train_loader)}')
print(f'test batch : {len(test_loader)}')
print(f'dev batch : {len(dev_loader)}')

sample = train_dataset[0]
print("input_ids:", sample['input_ids'])
print("attention_mask:", sample['attention_mask'])
print("token_type_ids:", sample['token_type_ids'])
print("labels:", sample['labels'])

train batch : 528
test batch : 147
dev batch : 59
input_ids: tensor([   101,  39192,  51382,  10676,    117,  15055,  11057,  25743,  10835,
         32917,  10911,  10706,  14498,  24109,  30509,  10229,    117,  24604,
         35091,  11754,  32310,  20168,  68659,  84444,  14469,    118,  14469,
         11868,  10911,  17356,  14415, 110111,  27130,  10116,    119,    102,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,    

# modeling

In [ ]:
class Model(torch.nn.Module):
    def __init__(self, model_name, num_labels):
        super(Model, self).__init__()
        self.bert = BertForTokenClassification.from_pretrained(model_name,
                                                               num_labels=num_labels,
                                                               id2label=id2label,
                                                               label2id=label2id,
                                                               ignore_mismatched_sizes=True
                                                               )
    def forward(self, input_ids, attention_mask, token_type_ids, labels=None):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        return output

model = Model(model_name=model, num_labels=num_labels)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'total parameters: {total_params}')
print(f'trainable parameters: {train_params}')
print(f'num labels : {num_labels}')
print(f'device : {device}')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7323.47it/s]
BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok

total parameters: 177271307
trainable parameters: 177271307
num labels : 11
device : cuda


# training model

In [ ]:
# hyperparamter
epoch = 5
learning_rate = 2e-5
warmup_ratio = 0.1

optimizer = AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

total_steps = len(train_loader) * epoch
warmup_steps = int(total_steps * warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'epoch : {epoch}')
print(f'learning_rate : {learning_rate}')
print(f'warmup_ratio : {warmup_ratio}')


epoch : 5
learning_rate : 2e-05
warmup_ratio : 0.1


In [ ]:
class NERTrainer:
    def __init__(self, model, optimizer, scheduler, device, id2label):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.device          = device
        self.id2label        = id2label
        self.best_f1         = 0.0
        self.best_model_path = 'best_ner_model.pt'

    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0.0

        for batch in dataloader:
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            token_type_ids = batch['token_type_ids'].to(self.device)
            labels         = batch['labels'].to(self.device)

            self.optimizer.zero_grad()

            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                token_type_ids = token_type_ids,
                labels         = labels
            )

            loss = output.loss
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            self.scheduler.step()

        return total_loss / len(dataloader)

    def eval_epoch(self, loader):
        self.model.eval()
        total_loss = 0.0
        all_preds  = []
        all_labels = []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                labels         = batch['labels'].to(self.device)

                output = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    token_type_ids = token_type_ids,
                    labels         = labels
                )

                total_loss += output.loss.item()
                logits = output.logits
                preds  = torch.argmax(logits, dim=-1)

                for pred_seq, label_seq in zip(preds, labels):
                    pred_label = []
                    true_label = []

                    for p, l in zip(pred_seq, label_seq):
                        if l.item() == -100:
                            continue
                        pred_label.append(self.id2label[p.item()])
                        true_label.append(self.id2label[l.item()])

                    all_preds.append(pred_label)
                    all_labels.append(true_label)

        avg_loss  = total_loss / len(loader)
        precision = precision_score(all_labels, all_preds)
        recall    = recall_score(all_labels, all_preds)
        f1        = f1_score(all_labels, all_preds)

        return avg_loss, precision, recall, f1

    def save_best_model(self, f1):
        if f1 > self.best_f1:
            self.best_f1 = f1
            torch.save(self.model.state_dict(), self.best_model_path)
            return True
        return False

    def run(self, train_loader, dev_loader, epochs):
        for epoch in range(1, epochs + 1):
            train_loss = self.train_epoch(train_loader)
            dev_loss, dev_p, dev_r, dev_f1 = self.eval_epoch(dev_loader)

            print(f'Epoch {epoch}/{epochs}')
            print(f'  Train Loss : {train_loss:.4f}')
            print(f'  Dev Loss   : {dev_loss:.4f}')
            print(f'  Dev P      : {dev_p:.4f}')
            print(f'  Dev R      : {dev_r:.4f}')
            print(f'  Dev F1     : {dev_f1:.4f}')

            if self.save_best_model(dev_f1):
                print(f'  ✓ Best model saved (F1={self.best_f1:.4f})')

            print("-" * 50)

        print(f'\nTraining selesai. Best F1: {self.best_f1:.4f}')



epochs = 5

trainer = NERTrainer(
    model     = model,
    optimizer = optimizer,
    scheduler = scheduler,
    device    = device,
    id2label  = id2label
)

trainer.run(train_loader, dev_loader, epochs)

Epoch 1/5
  Train Loss : 0.0383
  Dev Loss   : 0.1669
  Dev P      : 0.7964
  Dev R      : 0.8977
  Dev F1     : 0.8440
  ✓ Best model saved (F1=0.8440)
--------------------------------------------------
Epoch 2/5
  Train Loss : 0.0130
  Dev Loss   : 0.1931
  Dev P      : 0.8022
  Dev R      : 0.8861
  Dev F1     : 0.8421
--------------------------------------------------
Epoch 3/5
  Train Loss : 0.0056
  Dev Loss   : 0.2270
  Dev P      : 0.8110
  Dev R      : 0.8977
  Dev F1     : 0.8522
  ✓ Best model saved (F1=0.8522)
--------------------------------------------------
Epoch 4/5
  Train Loss : 0.0027
  Dev Loss   : 0.2319
  Dev P      : 0.8218
  Dev R      : 0.9001
  Dev F1     : 0.8591
  ✓ Best model saved (F1=0.8591)
--------------------------------------------------
Epoch 5/5
  Train Loss : 0.0023
  Dev Loss   : 0.2319
  Dev P      : 0.8218
  Dev R      : 0.9001
  Dev F1     : 0.8591
--------------------------------------------------

Training selesai. Best F1: 0.8591


# evaluasi

In [ ]:
class NEREvaluator:
    def __init__(self, model, device, id2label, model_path):
        self.model      = model
        self.device     = device
        self.id2label   = id2label
        self.model_path = model_path

    def load_best_model(self):
        self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
        self.model.to(self.device)
        print(f"✓ Model loaded from {self.model_path}")

    def evaluate(self, loader):
        self.model.eval()
        all_preds  = []
        all_labels = []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                labels         = batch['labels'].to(self.device)

                output = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    token_type_ids = token_type_ids,
                    labels         = labels
                )

                logits = output.logits
                preds  = torch.argmax(logits, dim=-1)

                for pred_seq, label_seq in zip(preds, labels):
                    pred_tags = []
                    true_tags = []
                    for p, l in zip(pred_seq, label_seq):
                        if l.item() == -100:
                            continue
                        pred_tags.append(self.id2label[p.item()])
                        true_tags.append(self.id2label[l.item()])
                    all_preds.append(pred_tags)
                    all_labels.append(true_tags)

        return all_preds, all_labels

    def print_report(self, all_labels, all_preds):
        print("=" * 60)
        print("EVALUASI TEST SET")
        print("=" * 60)
        print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
        print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
        print(f"F1-Score  : {f1_score(all_labels, all_preds):.4f}")
        print("\nPer-Entity Report:")
        print("-" * 60)
        print(classification_report(all_labels, all_preds))


evaluator = NEREvaluator(
    model      = model,
    device     = device,
    id2label   = id2label,
    model_path = "best_ner_model.pt"
)

evaluator.load_best_model()
test_preds, test_labels = evaluator.evaluate(test_loader)
evaluator.print_report(test_labels, test_preds)

C:\Users\kuala\AppData\Local\Temp\ipykernel_21832\4080880945.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(self.model_path, map_l

✓ Model loaded from best_ner_model.pt
EVALUASI TEST SET
Precision : 0.9797
Recall    : 0.9889
F1-Score  : 0.9843

Per-Entity Report:
------------------------------------------------------------
              precision    recall  f1-score   support

    LOCATION       0.98      0.99      0.99      1019
ORGANIZATION       0.97      0.98      0.97       823
      PERSON       0.99      0.99      0.99      1496
    QUANTITY       0.97      0.99      0.98       480
        TIME       0.98      0.99      0.98       433

   micro avg       0.98      0.99      0.98      4251
   macro avg       0.98      0.99      0.98      4251
weighted avg       0.98      0.99      0.98      4251



# Analisis

##  analisis output / hasil prediksi
| Metric     | Dev Set (Epoch 4-5) | Test Set |
|------------|---------------------|----------|
| Precision  | 0.8218              | 0.9797   |
| Recall     | 0.9001              | 0.9889   |
| F1-Score   | 0.8591              | 0.9840   |

Ada gap yang cukup signifikan antara dev dan test F1 (~0.125). Ini agak tidak biasa — biasanya test lebih rendah dari dev. Kemungkinan distribusi test set lebih "bersih" atau lebih mirip dengan pola training dibanding dev set dan jumlah epoch yang terlalu sedikit.

##  Contoh Entitas yang Dikenali dengan Benar
- PERSON — F1: 0.99 (tertinggi)

Nama orang dalam bahasa Indonesia biasanya punya pola kapital yang konsisten → mudah dikenali BERT
Dataset NerUGM mayoritas berita politik/sosial, jadi nama tokoh publik sering muncul di training → model punya representasi yang kuat

- LOCATION — F1: 0.99

Nama tempat juga punya sinyal kapital yang kuat
Konteks kalimat berita biasanya sangat informatif untuk lokasi ("di Jakarta", "ke Bandung")

- QUANTITY — F1: 0.98

Angka + satuan → pola leksikal yang sangat konsisten, BERT mudah menangkap
## Entitas yang Paling Rentan Error
- ORGANIZATION — F1: 0.97 (terendah)
Ini entitas paling susah dan hampir selalu jadi titik lemah di semua model NER bahasa Indonesia. Kenapa:

Nama organisasi sering overlap dengan lokasi. Contoh: "Universitas Indonesia" → bisa dibaca sebagai ORG atau LOC tergantung konteks
Singkatan seperti "DPR", "KPK", "DPRD" → perlu konteks untuk disambiguasi dengan nama lain
Organisasi baru atau informal tidak ada di vocab BERT

- TIME — F1: 0.98 (kedua terendah)

Ekspresi waktu dalam bahasa Indonesia sangat bervariasi: "kemarin", "Senin lalu", "pada 2019", "awal bulan ini"
Beberapa ekspresi temporal overlap dengan QUANTITY (misalnya "tiga hari" → TIME atau QUANTITY?)

##  Pola Error yang Kemungkinan Terja

a. B/I tag confusion
Recall lebih tinggi dari precision di semua epoch dev (P: 0.82, R: 0.90) → model cenderung over-predict span entitas. Artinya model sering memulai entitas di tempat yang benar tapi memperpanjang spannya terlalu jauh, atau sebaliknya memecah satu entitas multi-kata jadi dua span terpisah.
contoh

contoh : rayzan fazri -> person di tokenisasi dan di beri lable menjadi ['rayzan']-> person, ['fazri'] -> person
